#### ── Notebook Info ──────────────────────────────────────
#### Repo   : github.com/ShamsherSingh85/2026_AIE
#### Phase  : 1 | Week : 1 | Day : 1
#### Topic  : The Modern LLM Landscape
#### Saved  : File → Save a copy in GitHub
#### ───────────────────────────────────────────────────────

# Day 1 — The Modern LLM Landscape
**Date:** 9 March 2026 | **Phase:** 1 | **Week:** 1
**Topic:** Transformer architecture, tokenisation, HuggingFace basics

## Agenda
- How transformers relate to my existing LSTM/attention knowledge
- GPT vs BERT architecture
- HuggingFace transformers library basics


In [ ]:
!pip install transformers torch
!pip install nbstripout

## 1. Load a model and generate text

In [ ]:
from transformers import pipeline

# The simplest way to use any LLM — the pipeline API
# This downloads a small GPT-2 model (~500MB)
generator = pipeline('text-generation', model = 'gpt2')

prompt = 'The key to become a great AI Engineer is '

result = generator(prompt, max_new_tokens = 100, num_return_sequences = 1)
print(result[0]['generated_text'])

## 2. Understand tokenisation

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')

sentences = ["I am training to become an AI Engineer by Oct 2026",
        "Machine learning is a subset of artificial intelligence",
        "LLMs use transformer architecture with self-attention",
        "I will be an AI Engineer by October 2026"]

for sentence in sentences:
    tokens = tokenizer.encode(sentence)
    decoded = [tokenizer.decode([token]) for token in tokens]
    print(f'Total Number of tokens {len(tokens)}')
    print('Token IDs : ', tokens)
    print('Tokens : ', decoded)

## 3. See Attention in action conceptually

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions = True)

text = 'The AI Engineer built a great product'
inputs = tokenizer(text, return_tensors = 'pt')

with torch.no_grad():
    outputs = model(**inputs)

attention = outputs.attentions

print(f'Number of attention layers = {len(attention)}')
print(f'Shape of one attention layer = {attention[0].shape}')

## 4. Token Budget Calculator

Every AI Engineer needs to know this. Real-world task: before sending text to an LLM API, you need to know:

- How many tokens is it?
- How much will it cost?
- Is it within the context limit?

In [ ]:
from transformers import AutoTokenizer

def token_budget(texts, model = 'gpt2', cost_per_1k = 0.002, context_limit = 20):
  """
  Analyses token usage for a list of texts.

  Args:
      texts         : list of strings to analyse
      model         : tokenizer to use
      cost_per_1k   : cost in USD per 1000 tokens
      context_limit : max tokens the model accepts

  Returns:
      Prints a summary table + total stats
  """

  tokenizer = AutoTokenizer.from_pretrained(model)

  total_tokens = 0
  total_words = 0

  print(f"\n{'='*95}")
  print(f"{'Text':<35} {'Words':>6} {'Tokens':>8} {'Tok/Word':>10} {'Cost($)':>10} {'Context%':>10} {'Status':>10}")
  print(f"{'='*95}")

  for text in texts:

    tokens = tokenizer.encode(text)
    token_count = len(tokens)
    word_count = len(text.split())
    ratio = round(token_count / word_count, 2) if word_count > 0 else 0
    cost = (token_count / 1000) * cost_per_1k
    pct = (token_count / context_limit) * 100
    status = "⚠️ WARN" if pct > 80 else "✅ OK"
    label = text[:32] + "..." if len(text) > 32 else text

    print(
        f"{label:<35} "
        f"{word_count:>6} "
        f"{token_count:>8} "
        f"{ratio:>10} "
        f"{cost:>10.5f} "
        f"{pct:>9.2f}% "
        f"{status:>10}"
    )

    total_tokens += token_count
    total_words += word_count

  total_cost = (total_tokens / 1000) * cost_per_1k

  print(f"{'='*95}")
  print(f"{'TOTAL':<35} {total_words:>6} {total_tokens:>8} {'':>10} {total_cost:>10.5f}")
  print(f"{'='*95}\n")


# ── Test it with these inputs ──────────────────────────────────
texts = [
    "Explain what a transformer model is in simple terms.",
    "I am training to become an AI Engineer by October 2026.",
    "LLMs use self-attention mechanisms to process sequences.",
    "Machine learning is a subset of artificial intelligence.",
    "Write a Python function to calculate fibonacci numbers.",
    "In the heart of an old village stood a massive banyan tree. Its roots spread wide and its branches reached high in the sky",
    "Data fetching across the internet is unpredictable. We use async/await to pause the code until the data arrives, and try/catch to prevent the entire application from crashing if the server is down",
    "Its been a very long and tiring day. Recommend me some good music to rejuvenate myself and lighten my mood before I sleep"
]

token_budget(texts)